# 🩺 Diabetes Risk Classifier
Predicts diabetes status: **No Diabetes**, **Prediabetes**, or **Diabetes** using a Random Forest model.

---
### Steps:
1. Install dependencies
2. Upload your dataset
3. Train the model
4. Evaluate & save

## 1. Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy joblib -q

## 2. Upload Dataset
Upload your CSV file (`diabetes_012_health_indicators_BRFSS2021.csv`) when prompted.

> Alternatively, you can download it from Kaggle: [BRFSS 2021 Dataset](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
import pandas as pd

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"✅ Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 3. Configuration

In [ ]:
# ---- Config ----
TARGET_COL  = "Diabetes_012"
CLASS_NAMES = {0: "No_Diabetes", 1: "Prediabetes", 2: "Diabetes"}
TEST_RATIO  = 0.2
MAX_ROWS    = 60_000   # set None to use all rows
SEED        = 42
# ----------------

## 4. Preprocess Data

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def build_xy(df):
    if TARGET_COL not in df.columns:
        raise ValueError(f"Missing target column: {TARGET_COL}")
    y = pd.to_numeric(df[TARGET_COL], errors="coerce").astype("Int64")
    X = df.drop(columns=[TARGET_COL]).copy()
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    mask = y.notna()
    return X.loc[mask], y.loc[mask].astype(int)

def stratified_sample(X, y, max_rows, seed):
    if max_rows is None or len(X) <= max_rows:
        return X, y
    _, X_s, _, y_s = train_test_split(X, y, train_size=max_rows, random_state=seed, stratify=y)
    return X_s.reset_index(drop=True), y_s.reset_index(drop=True)

X, y = build_xy(df)
X, y = stratified_sample(X, y, MAX_ROWS, SEED)

print(f"Features : {X.shape[1]}")
print(f"Samples  : {len(X):,}")
print("\nClass distribution:")
for cls, cnt in zip(*np.unique(y, return_counts=True)):
    print(f"  {CLASS_NAMES.get(int(cls), cls):15s} → {cnt:,}")

## 5. Train the Model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=SEED, stratify=y
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("clf",     RandomForestClassifier(
        n_estimators=120,
        random_state=SEED,
        class_weight={0: 1.0, 1: 4.0, 2: 2.0},
        n_jobs=-1,
    )),
])

print("Training... (this may take a minute)")
pipe.fit(X_train, y_train)
print("✅ Training complete!")

## 6. Evaluate

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = pipe.predict(X_test)

acc     = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
cm      = confusion_matrix(y_test, y_pred)
labels  = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]

print(f"Accuracy  : {acc:.4f}")
print(f"Macro F1  : {macro_f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=labels))

# Confusion matrix heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

## 7. Save & Download the Model

In [ ]:
import joblib

classes = sorted(int(c) for c in np.unique(y))
model_bundle = {
    "type"              : "sklearn_pipeline_random_forest",
    "title"             : "Diabetes Risk Classifier",
    "description"       : "Predicts diabetes status: no diabetes, prediabetes, or diabetes.",
    "pipeline"          : pipe,
    "features"          : list(X.columns),
    "classes"           : [{"value": c, "name": CLASS_NAMES.get(c, f"Class_{c}")} for c in classes],
    "negativeClassValue": 0,
    "metrics"           : {
        "testRatio"        : TEST_RATIO,
        "testCount"        : len(y_test),
        "accuracy"         : acc,
        "macroF1"          : macro_f1,
        "confusionMatrix"  : cm.tolist(),
    },
}

MODEL_PATH = "diabetes_model.joblib"
joblib.dump(model_bundle, MODEL_PATH)
print(f"✅ Model saved to {MODEL_PATH}")

# Download to your local machine
files.download(MODEL_PATH)